In [ ]:
!pip install -q langchain-text-splitters chromadb sentence-transformers groq google-genai

In [ ]:
import os
import getpass

def read_api_key(name):
    value = os.environ.get(name, "").strip()
    if not value:
        try:
            from google.colab import userdata
            value = (userdata.get(name) or "").strip()
        except Exception:
            value = ""
    if not value:
        value = getpass.getpass(f"{name} (press Enter to skip): ").strip()
    return value

GROQ_API_KEY = read_api_key("GROQ_API_KEY")
GEMINI_API_KEY = read_api_key("GEMINI_API_KEY")

print("Groq key provided:", bool(GROQ_API_KEY))
print("Gemini key provided:", bool(GEMINI_API_KEY))


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions

source_documents = {
    "solar_basics.txt": """Solar Energy
Solar panels convert sunlight into electrical energy using photovoltaic cells. Output depends on sunlight intensity, panel orientation, shading, and temperature.
A charge controller regulates power from panels before it reaches a battery. Battery storage can supply energy when solar production is low or unavailable.
""",
    "wind_basics.txt": """Wind Energy
Wind turbines convert the kinetic energy of moving air into electricity. Turbine output depends on wind speed, rotor design, and site conditions.
A wind-energy installation needs suitable siting, maintenance, grid integration, and safety planning.
""",
    "microgrid_notes.txt": """Hybrid Renewable Systems
A hybrid renewable system may combine solar panels, wind turbines, batteries, and a controller. Combining sources can improve supply availability when sunlight and wind conditions vary.
Energy monitoring helps operators understand generation, battery state, and demand. Protection circuits and correctly rated components are important for safe operation.
"""
}

splitter = RecursiveCharacterTextSplitter(
    chunk_size=260, chunk_overlap=35,
    separators=["\n\n", "\n", ". ", " ", ""]
)

all_chunks, all_metadata, all_ids = [], [], []
for source, content in source_documents.items():
    for index, chunk in enumerate(splitter.split_text(content)):
        all_chunks.append(chunk)
        all_metadata.append({"source": source, "chunk_index": index})
        all_ids.append(f"{source.replace('.txt', '')}_{index}")

client = chromadb.Client()
embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
collection = client.get_or_create_collection(
    name="renewable_rag_knowledge",
    embedding_function=embedding_function
)
collection.add(ids=all_ids, documents=all_chunks, metadatas=all_metadata)

print("Indexed chunks:", collection.count())


In [ ]:
def retrieve_context(query, n_results=4, similarity_threshold=0.25):
    result = collection.query(
        query_texts=[query],
        n_results=min(n_results, collection.count()),
        include=["documents", "metadatas", "distances"]
    )

    retrieved = []
    for text, meta, distance in zip(
        result["documents"][0],
        result["metadatas"][0],
        result["distances"][0]
    ):
        similarity = 1 - float(distance)
        if similarity >= similarity_threshold:
            retrieved.append({
                "text": text,
                "source": meta["source"],
                "similarity": similarity
            })
    return retrieved

test_query = "How can stored energy be used when solar production is low?"
for item in retrieve_context(test_query):
    print(f"{item['similarity']:.3f} | {item['source']} | {item['text']}")


In [ ]:
SYSTEM_PROMPT = (
    "You answer questions using only the supplied context. "
    "If the context does not contain the answer, say: "
    "'I don't know based on the provided documents.' "
    "Keep the answer concise and name the source file or files used."
)

def build_user_prompt(query, chunks):
    context_text = "\n\n".join(
        f"[Source: {item['source']}]\n{item['text']}"
        for item in chunks
    )
    return f"Context:\n{context_text}\n\nQuestion: {query}\n\nAnswer:"


In [ ]:
def ask_groq(prompt):
    if not GROQ_API_KEY:
        return "Groq skipped: no API key provided."
    from groq import Groq
    client = Groq(api_key=GROQ_API_KEY)
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.1
    )
    return response.choices[0].message.content


In [ ]:
def ask_gemini(prompt):
    if not GEMINI_API_KEY:
        return "Gemini skipped: no API key provided."
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=GEMINI_API_KEY)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0.1
        )
    )
    return response.text or "The model returned no text."


In [ ]:
def rag_answer(query, provider="groq", show_sources=True):
    chunks = retrieve_context(query)
    if not chunks:
        return "I don't know based on the provided documents. No relevant context was retrieved."

    prompt = build_user_prompt(query, chunks)
    if provider.lower() == "groq":
        answer = ask_groq(prompt)
    elif provider.lower() == "gemini":
        answer = ask_gemini(prompt)
    else:
        raise ValueError("provider must be 'groq' or 'gemini'")

    if show_sources:
        sources = sorted({item["source"] for item in chunks})
        return answer + "\n\nSources: " + ", ".join(sources)
    return answer


In [ ]:
question = "What components may be included in a hybrid renewable system?"
print(rag_answer(question, provider="groq"))


In [ ]:
question = "How do solar and wind systems generate electricity, and why might they be combined?"
print(rag_answer(question, provider="gemini"))


In [ ]:
question = "What is the capital city of Brazil?"
print(rag_answer(question, provider="groq"))
